# Đáp Án Mẫu — Bộ Dữ Liệu Tech Salary (mở rộng, tùy chọn)

Notebook này dành cho **người tham gia**, luyện tập thêm với bộ dữ liệu mở rộng `data/tech_salary.duckdb` (xem `schemas/TECH_SALARY_DATASET_SCHEMA.md`), độc lập hoàn toàn với `data/workshop.duckdb`. Cùng cấu trúc 3 cấp độ Cơ bản / Trung cấp / Nâng cao như `exercises_answer_key_bank_schema.ipynb`.

3 câu Cơ bản đầu tiên (B1–B3) khớp với 3 câu hỏi mẫu ở Bước 7 của `../exercises.md` — dùng để đối chiếu kết quả sau khi mở rộng skill của bạn sang database thứ hai. Các câu còn lại (B4–B5, toàn bộ Trung cấp và Nâng cao) là bài tập thêm, không có trong `exercises.md`, dùng để luyện tập sâu hơn nếu bạn đã xong Bước 7.

Vì `global_tech_market_2026` và `usajobs_tech_roles_2026` không có khóa ngoại nối với nhau (xem "Lưu ý & điểm đặc biệt" trong từ điển dữ liệu), "Trung cấp" ở đây không phải là JOIN hai bảng như ở bộ dữ liệu ngân hàng — mà là kết hợp `UNION ALL` hai bảng cùng schema, thêm điều kiện lọc/phân nhóm phức tạp hơn.

Với các câu **Nâng cao**, không có một câu SQL "đúng" duy nhất — SQL bên dưới chỉ là **một cách hợp lý** để trả lời.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 120)

con = duckdb.connect("../data/tech_salary.duckdb", read_only=True)
con.sql("SELECT table_name, estimated_size AS so_dong FROM duckdb_tables()").df()


,table_name,so_dong
0,global_tech_market_2026,12003
1,usajobs_tech_roles_2026,2997


## Cấp độ Cơ bản — một bảng (hoặc `UNION ALL` đơn giản), tổng hợp cơ bản

### B1. Mức lương trung bình cho vị trí Data Scientist là bao nhiêu?

Tính trung bình `(salary_min_usd + salary_max_usd) / 2`, gộp cả hai bảng.

In [2]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT
    job_title,
    ROUND(AVG((salary_min_usd + salary_max_usd) / 2.0), 0) AS luong_trung_binh_usd,
    COUNT(*) AS so_tin_dang
FROM all_jobs
WHERE job_title = 'Data Scientist'
GROUP BY job_title
''').df()


,job_title,luong_trung_binh_usd,so_tin_dang
0,Data Scientist,122723.0,769


### B2. 3 tổ hợp công nghệ (`tech_stack`) xuất hiện trong nhiều tin đăng nhất là gì?

In [3]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT tech_stack, COUNT(*) AS so_tin_dang
FROM all_jobs
GROUP BY tech_stack
ORDER BY so_tin_dang DESC
LIMIT 3
''').df()


,tech_stack,so_tin_dang
0,"Java, Spring Boot, Kafka, PostgreSQL",1555
1,"Rust, WebAssembly, System Architecture",1548
2,"Ruby on Rails, Redis, Heroku",1528


### B3. So sánh mức lương tối đa trung bình giữa tin đăng từ USAJOBS và toàn bộ thị trường tech nói chung.

In [4]:
con.sql('''
SELECT 'USAJOBS' AS nguon, ROUND(AVG(salary_max_usd), 0) AS luong_toi_da_trung_binh_usd
FROM usajobs_tech_roles_2026
UNION ALL
SELECT 'Toan bo thi truong (global_tech_market_2026)' AS nguon, ROUND(AVG(salary_max_usd), 0) AS luong_toi_da_trung_binh_usd
FROM global_tech_market_2026
''').df()


,nguon,luong_toi_da_trung_binh_usd
0,USAJOBS,143374.0
1,Toan bo thi truong (global_tech_market_2026),141565.0


### B4. Số lượng tin đăng theo từng chức danh (`job_title`) trong `global_tech_market_2026`, sắp xếp giảm dần?

In [5]:
con.sql('''
SELECT job_title, COUNT(*) AS so_tin_dang
FROM global_tech_market_2026
GROUP BY job_title
ORDER BY so_tin_dang DESC
''').df()


,job_title,so_tin_dang
0,Backend Developer,635
1,DevOps Engineer,627
2,Frontend Developer,625
3,Site Reliability Engineer,622
4,Lead Data Scientist,620
5,Data Scientist,618
6,Analytics Engineer,606
7,Full Stack Developer,604
8,NLP Engineer,601
9,Technical Program Manager,595


### B5. Mức lương tối đa trung bình (`salary_max_usd`) theo từng địa điểm (`location`) trong `global_tech_market_2026`?

In [6]:
con.sql('''
SELECT location, ROUND(AVG(salary_max_usd), 0) AS luong_toi_da_trung_binh
FROM global_tech_market_2026
GROUP BY location
ORDER BY luong_toi_da_trung_binh DESC
''').df()


,location,luong_toi_da_trung_binh
0,"Seattle, WA, USA",195254.0
1,"New York, NY, USA",193840.0
2,"San Francisco, CA, USA",189818.0
3,"Toronto, Canada",149886.0
4,"Sydney, Australia",149876.0
5,"Austin, TX, USA",149129.0
6,Singapore,147711.0
7,"Paris, France",146532.0
8,Remote - Global,134549.0
9,Remote - US Only,132799.0


## Cấp độ Trung cấp — kết hợp `UNION ALL` cả hai bảng, phân nhóm/lọc theo nhiều tiêu chí

### I1. Mức lương trung bình và số tin đăng có khác nhau giữa vị trí Remote và vị trí tại văn phòng không?

"Remote" = `location` bắt đầu bằng `Remote` (ví dụ `Remote - Global`, `Remote - EMEA`).

In [7]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT
    CASE WHEN location LIKE 'Remote%' THEN 'Remote' ELSE 'Tai van phong' END AS loai_dia_diem,
    COUNT(*) AS so_tin_dang,
    ROUND(AVG((salary_min_usd + salary_max_usd) / 2.0), 0) AS luong_trung_binh_usd
FROM all_jobs
GROUP BY loai_dia_diem
ORDER BY so_tin_dang DESC
''').df()


,loai_dia_diem,so_tin_dang,luong_trung_binh_usd
0,Tai van phong,11254,124129.0
1,Remote,3746,113906.0


### I2. Trong số các tin đăng có `tech_stack` chứa "Python", mức lương trung bình theo từng chức danh (`job_title`) là bao nhiêu?

In [8]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT
    job_title,
    COUNT(*) AS so_tin_dang,
    ROUND(AVG((salary_min_usd + salary_max_usd) / 2.0), 0) AS luong_trung_binh_usd
FROM all_jobs
WHERE tech_stack LIKE '%Python%'
GROUP BY job_title
ORDER BY luong_trung_binh_usd DESC
''').df()


,job_title,so_tin_dang,luong_trung_binh_usd
0,Lead Data Scientist,234,183425.0
1,Senior Data Scientist,227,180209.0
2,Staff Engineer,204,164310.0
3,Senior Software Engineer,238,159499.0
4,Cloud Architect,213,157457.0
5,Data Scientist,214,124683.0
6,AI Researcher,228,119798.0
7,Data Engineer,205,119762.0
8,Machine Learning Engineer,194,118273.0
9,Analytics Engineer,222,103462.0


### I3. Mức lương trung bình chênh lệch bao nhiêu giữa các vị trí cấp cao (chức danh có tiền tố Senior/Lead/Staff) và các vị trí còn lại?

In [9]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT
    CASE
        WHEN job_title LIKE 'Senior%' OR job_title LIKE 'Lead%' OR job_title LIKE 'Staff%' THEN 'Cap cao (Senior/Lead/Staff)'
        ELSE 'Cap con lai'
    END AS nhom_cap_bac,
    COUNT(*) AS so_tin_dang,
    ROUND(AVG((salary_min_usd + salary_max_usd) / 2.0), 0) AS luong_trung_binh_usd
FROM all_jobs
GROUP BY nhom_cap_bac
ORDER BY luong_trung_binh_usd DESC
''').df()


,nhom_cap_bac,so_tin_dang,luong_trung_binh_usd
0,Cap cao (Senior/Lead/Staff),2999,171989.0
1,Cap con lai,12001,108978.0


### I4. Trong `usajobs_tech_roles_2026`, cơ quan (`company_name`) nào trả mức lương trung bình cao nhất?

In [10]:
con.sql('''
SELECT
    company_name,
    COUNT(*) AS so_tin_dang,
    ROUND(AVG((salary_min_usd + salary_max_usd) / 2.0), 0) AS luong_trung_binh_usd
FROM usajobs_tech_roles_2026
GROUP BY company_name
ORDER BY luong_trung_binh_usd DESC
''').df()


,company_name,so_tin_dang,luong_trung_binh_usd
0,Federal Bureau of Investigation,168,128653.0
1,NASA,166,127673.0
2,European Space Agency,178,127056.0
3,Department of Defense,162,125429.0
4,Umbrella Corp,177,124888.0
5,HealthAI,169,124678.0
6,Globex,161,124058.0
7,GlobalSystems Inc,184,123582.0
8,EduTech Global,162,123057.0
9,Initech,203,122679.0


### I5. 5 chức danh (`job_title`) nào có tỷ lệ tin đăng Remote cao nhất?

In [11]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT
    job_title,
    COUNT(*) AS tong_tin_dang,
    SUM(CASE WHEN location LIKE 'Remote%' THEN 1 ELSE 0 END) AS so_tin_remote,
    ROUND(100.0 * SUM(CASE WHEN location LIKE 'Remote%' THEN 1 ELSE 0 END) / COUNT(*), 2) AS ty_le_remote_pct
FROM all_jobs
GROUP BY job_title
ORDER BY ty_le_remote_pct DESC
LIMIT 5
''').df()


,job_title,tong_tin_dang,so_tin_remote,ty_le_remote_pct
0,Full Stack Developer,731,206.0,28.18
1,Cybersecurity Analyst,717,201.0,28.03
2,Staff Engineer,757,211.0,27.87
3,NLP Engineer,766,211.0,27.55
4,Senior Software Engineer,741,199.0,26.86


## Cấp độ Nâng cao — CTE/window function, suy luận nghiệp vụ

Nhắc lại: các câu này không có một đáp án SQL duy nhất. SQL dưới đây là ví dụ tham khảo, có nêu rõ giả định.

### A1. Xếp hạng độ hấp dẫn của từng chức danh

Giả định về cách tính điểm hấp dẫn (0–100), chỉ là một cách hợp lý trong nhiều cách — kết hợp:
- 70% trọng số: mức lương trung bình, chuẩn hóa min-max trong toàn bộ chức danh
- 30% trọng số: số lượng tin đăng (thị trường việc làm rộng hơn), cũng chuẩn hóa min-max

In [12]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
),
stats AS (
    SELECT
        job_title,
        COUNT(*) AS so_tin_dang,
        ROUND(AVG((salary_min_usd + salary_max_usd) / 2.0), 0) AS luong_trung_binh_usd
    FROM all_jobs
    GROUP BY job_title
)
SELECT
    *,
    ROUND(
        100.0 * (luong_trung_binh_usd - MIN(luong_trung_binh_usd) OVER ())
            / NULLIF(MAX(luong_trung_binh_usd) OVER () - MIN(luong_trung_binh_usd) OVER (), 0) * 0.7
        + 100.0 * (so_tin_dang - MIN(so_tin_dang) OVER ())
            / NULLIF(MAX(so_tin_dang) OVER () - MIN(so_tin_dang) OVER (), 0) * 0.3
    , 2) AS diem_hap_dan
FROM stats
ORDER BY diem_hap_dan DESC
''').df()


,job_title,so_tin_dang,luong_trung_binh_usd,diem_hap_dan
0,Lead Data Scientist,778,181310.0,93.96
1,Senior Data Scientist,723,181825.0,72.40
2,Staff Engineer,757,163650.0,70.58
3,Senior Software Engineer,741,161123.0,62.04
4,Cloud Architect,732,160568.0,57.97
5,Data Scientist,769,122723.0,40.66
6,Data Engineer,750,120084.0,30.82
7,Backend Developer,792,99318.0,30.01
8,DevOps Engineer,790,100029.0,29.81
9,Machine Learning Engineer,731,121127.0,24.11


### A2. Trong số các tổ hợp công nghệ (`tech_stack`) có ít nhất 500 tin đăng, tổ hợp nào trả lương trung bình cao nhất?

In [13]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT
    tech_stack,
    COUNT(*) AS so_tin_dang,
    ROUND(AVG((salary_min_usd + salary_max_usd) / 2.0), 0) AS luong_trung_binh_usd
FROM all_jobs
GROUP BY tech_stack
HAVING COUNT(*) >= 500
ORDER BY luong_trung_binh_usd DESC
''').df()


,tech_stack,so_tin_dang,luong_trung_binh_usd
0,"C++, CUDA, Computer Vision",1482,122770.0
1,"Go, Kubernetes, Docker, GCP",1480,122259.0
2,"Python, Airflow, Snowflake, dbt",1500,121881.0
3,"Ruby on Rails, Redis, Heroku",1528,121686.0
4,"TypeScript, Angular, Firebase",1495,121625.0
5,"JavaScript, React, Node.js, MongoDB",1450,121561.0
6,"Java, Spring Boot, Kafka, PostgreSQL",1555,121468.0
7,"Python, SQL, TensorFlow, PyTorch",1486,121210.0
8,"Python, Pandas, Scikit-Learn, AWS",1476,121050.0
9,"Rust, WebAssembly, System Architecture",1548,120301.0


### A3. Trong nhóm 10% tin đăng có `salary_max_usd` cao nhất, chức danh (`job_title`) nào chiếm đa số?

Dùng `PERCENT_RANK()` để lấy đúng phân vị 10% trên cùng, theo `salary_max_usd`.

In [14]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
),
ranked AS (
    SELECT *, PERCENT_RANK() OVER (ORDER BY salary_max_usd) AS pct_rank
    FROM all_jobs
),
top10 AS (
    SELECT * FROM ranked WHERE pct_rank >= 0.90
)
SELECT job_title, COUNT(*) AS so_tin_trong_top10pct
FROM top10
GROUP BY job_title
ORDER BY so_tin_trong_top10pct DESC
''').df()


,job_title,so_tin_trong_top10pct
0,Lead Data Scientist,390
1,Senior Data Scientist,366
2,Staff Engineer,221
3,Senior Software Engineer,210
4,Cloud Architect,197
5,Data Scientist,35
6,AI Researcher,28
7,Data Engineer,27
8,Machine Learning Engineer,26


### A4. Với mỗi nguồn tin đăng (`source`), tổ hợp công nghệ (`tech_stack`) phổ biến nhất là gì?

Dùng `ROW_NUMBER()` phân vùng (`PARTITION BY`) theo `source` để lấy đúng 1 tổ hợp phổ biến nhất mỗi nguồn.

In [15]:
con.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
),
counts AS (
    SELECT source, tech_stack, COUNT(*) AS so_tin_dang
    FROM all_jobs
    GROUP BY source, tech_stack
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY source ORDER BY so_tin_dang DESC) AS rn
    FROM counts
)
SELECT source, tech_stack, so_tin_dang
FROM ranked
WHERE rn = 1
ORDER BY so_tin_dang DESC
''').df()


,source,tech_stack,so_tin_dang
0,HackerNews,"Java, Spring Boot, Kafka, PostgreSQL",341
1,Glassdoor_Scrape,"Python, SQL, TensorFlow, PyTorch",334
2,LinkedIn_Proxy,"Ruby on Rails, Redis, Heroku",324
3,Company_Career_Page,"TypeScript, Angular, Firebase",322
4,USAJOBS,"C++, CUDA, Computer Vision",316
